# Physiological, Metabolic, Elemental, Pigment, Vitamin, and Antioxidant Measurements for Odontella aurita Under Eight Light Spectra Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.t08f-8v6s/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR^2 dataset using `mlcroissant`. This section demonstrates loading the dataset schema, extracting metadata, and confirming the dataset description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.t08f-8v6s/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as a single object
metadata = dataset.metadata

print(f"\nDataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print("\nPublished Date:", getattr(metadata, 'datePublished', 'N/A'))
print("\nLicense:", getattr(metadata, 'license', 'N/A'))

## 2. Data Overview
List available record sets and their `@id`s. Also enumerate fields (columns) for each record set. Entities are referenced by their `@id`, as per best practices for Croissant datasets.

In [ ]:
# List all record sets in the dataset by their @id
record_sets = dataset.record_sets

print('Record Sets (@id):')
for rs in record_sets:
    print(' -', rs['@id'], '|', rs.get('name','[no name]'))

# For each record set, list its fields/columns by @id and name
print('\nFields/Columns for each Record Set:')
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} ({rs.get('name','[no name]')})")
    columns = rs.get('field', rs.get('column', []))
    # Some schemas use 'field', some 'column'
    for field in columns:
        if isinstance(field, dict):
            print("  -", field.get('@id'), "|", field.get('name', '[no name]'))
        else:
            print("  -", field) # In case only @id is given

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. Use only `@id` to reference record sets and fields. All DataFrames are indexed by their record set `@id`. We process each record set sequentially and preview column names and sample records.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nColumns for Record Set {rs_id}:")
            print(df.columns.tolist())
            print(f"Sample records for {rs_id}:")
            display(df.head(2))
    except Exception as e:
        print(f"\nCould not load records for {rs_id} ({str(e)})")

# Choose the first available record set for further processing
record_set_to_use = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Perform basic EDA, including filtering records based on a numeric field, normalizing that field, and grouping if possible—always by `@id`.

Below, we dynamically select a numeric field to use based on the fields present in the chosen record set.

In [ ]:
# EDA on the chosen record set
df = dataframes.get(record_set_to_use)
if df is None:
    print("No records to analyze.")
else:
    # List numeric fields by their @id
    numeric_types = ['Integer', 'Float', 'Number']
    def get_numeric_field(rs):
        columns = rs.get('field', rs.get('column', []))
        for field in columns:
            if isinstance(field, dict) and field.get('dataType'):
                # The dataType field may be a @id or vocab
                dt = field['dataType']
                if isinstance(dt, dict):
                    dt = dt.get('@id')
                # Accept vocab-like or schema-like types
                if any(t.lower() in str(dt).lower() for t in numeric_types):
                    return field['@id']
        # fallback: try for a numeric column by dtype
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                return c
        return None

    numeric_field_id = get_numeric_field(next(rs for rs in record_sets if rs['@id']==record_set_to_use))
    group_field_id = None
    # Try to select a groupable (categorical) field
    columns = next(rs for rs in record_sets if rs['@id']==record_set_to_use).get('field', [])
    for field in columns:
        if isinstance(field, dict) and field.get('dataType'):
            dt = field['dataType']
            if isinstance(dt, dict):
                dt = dt.get('@id')
            # Choose the first Text or categorical field
            if 'Text' in str(dt) or 'String' in str(dt):
                group_field_id = field['@id']
                break
    if not group_field_id:
        # fallback: select first object dtype
        for c in df.columns:
            if pd.api.types.is_object_dtype(df[c]) and c != numeric_field_id:
                group_field_id = c
                break
    # Filtering/normalization
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())

        # Grouping
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("\nNo suitable group field found for grouping.")
    else:
        print("No numeric field found for analysis.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and if applicable, a scatter plot grouping by the chosen categorical field.
All visualizations use `@id` references for the involved fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # If grouping field available, plot mean values per group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values()
        group_means.plot(kind='bar')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.show()

## 6. Conclusion
In this notebook, we used `mlcroissant` to load the FAIR^2 dataset and explored its rich set of record sets and fields using unique `@id` references for each entity. We performed basic data extraction, filtering, normalization, grouping, and visualization. This workflow demonstrates how Croissant schemas and `mlcroissant` enable reproducible, machine-readable data science on complex, multi-level datasets.

For further analysis, extend this notebook by:
- Exploring additional record sets and their relationships (using their `@id`)
- Applying advanced statistical techniques or modeling
- Integrating metadata-driven field descriptions
- Saving processed data for downstream FAIR workflows